In [1]:
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd
from form990_parser import (
    parse_header,
    parse_return,
    get_org_type
)

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
xml_data_directory = Path(r'xml')

In [4]:
header_data = []
return_data = []
employee_data = []
contractor_data = []
expense_data = []
balance_sheet_data = []

for year_dir in xml_data_directory.iterdir():
    for period_dir in year_dir.iterdir():
        print(period_dir)
        counter = 0
        files_skipped = 0
        for xml_file in period_dir.glob('*.xml'):
            # Skip certain orgs and form types
            root = ET.parse(xml_file).getroot()
            org_type = get_org_type(root)
            if org_type not in ['501c3', '501c4', '527']:
                files_skipped += 1
                continue 

            if counter >= 50:
                break
            counter += 1

            # * Header
            header_content = parse_header(root, xml_file, org_type)
            header_data.append(header_content)

            # * Return
            if parse_return(root, 'tbd') is None:
                continue # Some files pertain to other types of returns (e.g., 990-T)
            return_content, employee_content, contractor_content, expense_content, balance_sheet_content = parse_return(root, 'tbd')
            return_data.append(return_content)
            
            # * Employee
            employee_data += employee_content
            
            # * Contractor
            contractor_data += contractor_content

            # * Expenses
            expense_data += expense_content
            
            # * Balance Sheet
            balance_sheet_data += balance_sheet_content
        # print(counter) # Uncomment when you want to know the actual number of files
        print(f"# Files skipped: {files_skipped}")

xml\2020\2020_1A
# Files skipped: 0
xml\2020\2020_3A
# Files skipped: 11
xml\2024\2024_01A
# Files skipped: 69
xml\2024\2024_02A
# Files skipped: 93
xml\2025\2025_01A
# Files skipped: 113
xml\2025\2025_02A
# Files skipped: 116
xml\2025\2025_03A
# Files skipped: 193
xml\2025\2025_04A
# Files skipped: 191
xml\2025\2025_05A
# Files skipped: 294
xml\2025\2025_05B
# Files skipped: 12
xml\2025\2025_06A
# Files skipped: 142
xml\2025\2025_07A
# Files skipped: 248
xml\2025\2025_08A
# Files skipped: 135
xml\2025\2025_09A
# Files skipped: 139
xml\2025\2025_10A
# Files skipped: 162
xml\2025\2025_11A
# Files skipped: 179
xml\2025\2025_11B
# Files skipped: 12
xml\2025\2025_11C
# Files skipped: 327
xml\2025\2025_11D
# Files skipped: 136
xml\2025\2025_12A
# Files skipped: 112
xml\2026\2026_1A
# Files skipped: 112
xml\2026\2026_2A
# Files skipped: 79
xml\2026\2026_3A
# Files skipped: 178
xml\2026\2026_4A
# Files skipped: 177


In [5]:
pd.DataFrame(header_data)

,FilerEIN,TaxPeriodBeginDate,TaxPeriodEndDate,TaxYear,ReturnType,FilerBusinessNameLine1Txt,FilerBusinessNameLine2Txt,FilerPhone,FilerAddressLine1Txt,FilerCity,...,PreparerCity,PreparerStateCode,PreparerZipCode,BusinessOfficerName,BusinessOfficerTitle,BusinessOfficerPhone,BusinessOfficerSignatureDate,org_type,file_name,NumOfficers
0,742044647,2018-07-01,2019-06-30,2018,990,NATIONAL JEWISH HEALTH,NaN,3033884461,1400 JACKSON STREET,DENVER,...,NaN,NaN,NaN,Christine Forkner,Chief Financial Officer,3033981004,2020-06-02,501c3,xml\2020\2020_3A\202011559349300711_public.xml,1
1,581416325,2019-01-01,2019-12-31,2019,990,HAWTREE VOLUNTEER FIRE DEPARTMENT INC,NaN,2522573288,PO BOX 116,WISE,...,HENDERSON,NC,27536,STEVE BARNEY,CHIEF,2524325386,2020-05-27,501c3,xml\2020\2020_3A\202011559349300716_public.xml,1
2,223139262,2018-07-01,2019-06-30,2018,990,RURAL DEVELOPMENT INC,NaN,4138639781,241 MILLERS FALLS ROAD,TURNERS FALLS,...,WESTBOROUGH,MA,01581,GINA GOVONI,EXECUTIVE DIRECTOR,4138639781,2020-05-06,501c3,xml\2020\2020_3A\202011559349300726_public.xml,1
3,363244264,2018-07-01,2019-06-30,2018,990,WILL COUNTY GOVERNMENTAL LEAGUE,NaN,8152547700,15905 S FREDERICK STREET Suite 107,PLAINFIELD,...,Oakbrook Terrace,IL,601815209,JOHN NOAK,PRESIDENT,8157293535,2020-05-15,501c4,xml\2020\2020_3A\202011559349300731_public.xml,1
4,043316917,2019-01-01,2019-12-31,2019,990,I'M STILL HERE FOUNDATION INC,NaN,7815690229,10 TOWER OFFICE PARK NO 317,WOBURN,...,BOSTON,MA,02109,JACQUELINE VISCHER,TREASURER,7815690229,2020-05-21,501c3,xml\2020\2020_3A\202011559349300736_public.xml,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1145,341577595,2024-07-01,2025-06-30,2024,990,STARK STATE COLLEGE FOUNDATION,NaN,3304946170,6200 Frank Ave NW,North Canton,...,NaN,NaN,NaN,Joseph Richards,Comptroller,3304946170,2026-03-05,501c3,xml\2026\2026_4A\202600869349300520_public.xml,1
1146,721362424,2025-01-01,2025-12-31,2025,990,NORCO CIVIC ASSOCIATION,NaN,9857649696,PO BOX 22,NORCO,...,NORCO,LA,70079,CLARENCE DUPEPE,TREASURER,9857649696,2026-03-24,501c4,xml\2026\2026_4A\202600869349300525_public.xml,1
1147,980142381,2025-01-01,2025-12-31,2025,990,BETHSHEAN MEXICO MISSION INC,NaN,7062555330,PO BOX 7391,ATHENS,...,HARTWELL,GA,30643,DAN GILLAND,SEC/TREASURER,NaN,2026-03-12,501c3,xml\2026\2026_4A\202600869349300535_public.xml,1
1148,631288598,2025-07-01,2025-09-30,2025,990,C A S A OF FLORENCE LAUDERDALE,NaN,2567650041,102 JACKSON COURT,SHEFFIELD,...,FLORENCE,AL,35630,HEATHER BEGLEY,DIRECTOR,2567650041,2026-03-11,501c3,xml\2026\2026_4A\202600869349300540_public.xml,1


In [6]:
pd.DataFrame(return_data).head()

,PrincipalOfficer,PrincipalOfficerAddress,PrincipalOfficerCity,PrincipalOfficerStateCode,PrincipalOfficerZipCode,GrossReceiptsAmt,ActivityOrMission,MissionDesc,FormationYear,VotingMembersGoverningBodyCnt,...,RelatedOrganizationsAmt,GovernmentGrantsAmt,AllOtherContributionsAmt,NoncashContributionsAmt,TotalContributionsAmt,TotalProgramServiceRevenueAmt,TotalRevenueColumnAmt,RelatedOrExemptFuncIncomeAmt,UnrelatedBusinessRevenueAmt,ExclusionAmt
0,Christine Forkner,1400 Jackson Street,Denver,CO,80206,326863373,National Jewish Health's mission since 1899 is...,National Jewish Health's mission since 1899 is...,1978,44,...,0,60946782,29837556,3056144,96842809,192033726,298202040,187160868,4759037,9439326
1,STEVE BARNEY,PO BOX 116,WISE,NC,27594,107616,TO PROVIDE FIRE DEPARTMENT SERVICES FOR THE CO...,TO PROVIDE FIRE DEPARTMENT SERVICES FOR THE CO...,1981,9,...,NaN,15249,75692,NaN,90941,NaN,103997,3630,0,9426
2,GINA GOVONI,241 MILLERS FALLS ROAD,TURNERS FALLS,MA,01376,301633,"IT IS THE MISSION OF RURAL DEVELOPMENT, INC. (...",IT IS THE MISSION OF RDI TO ADVANCE THE RIGHT ...,1991,8,...,NaN,NaN,6470,NaN,6470,292417,301633,294917,0,246
3,JOHN NOAK,15905 S FREDRICK STREET 107,PLAINFIELD,IL,60586,740025,GOVERNMENT SERVICE,THE WILL COUNTY GOVERNMENTAL LEAGUE IS ORGANIZ...,1983,9,...,NaN,152971,NaN,NaN,152971,472601,664134,472601,NaN,38562
4,JACQUELINE VISCHER,10 TOWER OFFICE PARK NO 317,WOBURN,MA,01801,18632,TO BRING ABOUT COMMUNITY TRANSFORMATION FOR TH...,TO BRING ABOUT COMMUNITY TRANSFORMATION FOR TH...,1995,6,...,NaN,NaN,12724,NaN,12724,5908,18632,5908,0,0


In [7]:
pd.DataFrame(employee_data)

,EIN,PersonNm,TitleTxt,AverageHoursPerWeekRt,AverageHoursPerWeekRltdOrgRt,IndividualTrusteeOrDirectorInd,ReportableCompFromOrgAmt,ReportableCompFromRltdOrgAmt,OtherCompensationAmt,OfficerInd,KeyEmployeeInd,HighestCompensatedEmployeeInd,FormerOfcrDirectorTrusteeInd,BusinessName,InstitutionalTrusteeInd
0,tbd,Jandel Allen-Davis MD,"Member, BOD",2,0,X,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,tbd,Sue Allon,"Member, BOD",2,0,X,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,tbd,Steve Arent,"Lifetime Member, BOD",0,0,X,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,tbd,Richard Baer,"Member, BOD",2,0,X,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,tbd,Geoffrey Barker,"Member, BOD",2,0,X,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12101,tbd,MARK WOOD,PRESIDENT,NaN,NaN,NaN,0,0,0,X,NaN,NaN,NaN,NaN,NaN
12102,tbd,STEPHEN BARNARD,PRESIDENT,1.00,NaN,X,0,0,0,X,NaN,NaN,NaN,NaN,NaN
12103,tbd,ANITA LEMOS,V.P.,1.00,NaN,X,0,0,0,X,NaN,NaN,NaN,NaN,NaN
12104,tbd,BRYAN GILES,TREASURER,1.00,NaN,X,0,0,0,X,NaN,NaN,NaN,NaN,NaN


In [8]:
pd.DataFrame(contractor_data)

,EIN,BusinessNameLine1Txt,AddressLine1Txt,AddressLine2Txt,CityNm,StateAbbreviationCd,ZIPCd,ServicesDesc,CompensationAmt,PersonNm,CountryCd,BusinessNameLine2Txt,ProvinceOrStateNm,ForeignPostalCd,ContractorAddress
0,tbd,Dimassimo,220 E 23rd Street,2nd Floor,New York,NY,10010,Advertising and Professional Fees,2203123,NaN,NaN,NaN,NaN,NaN,NaN
1,tbd,Siemens Medical Solutions USA Inc,51 Valley Stream Pkwy,NaN,Malvern,PA,19355,Equipment Maintenance Contract,1023832,NaN,NaN,NaN,NaN,NaN,NaN
2,tbd,HSS,MAIN,PO BOX 17033,Denver,CO,80217,Security Support,867767,NaN,NaN,NaN,NaN,NaN,NaN
3,tbd,ARUP Laboratories,MAIN,PO Box 27964,Salt Lake City,UT,84127,Lab Services,848779,NaN,NaN,NaN,NaN,NaN,NaN
4,tbd,Mindset Direct,12110 Sunset Hills Rd 600,NaN,Reston,VA,20190,Fundraising Servicses,612348,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
391,tbd,KENNETH MATTUS,1114 MAPLE STREET,NaN,SANTA MONICA,CA,90405,MUSIC PROGRAM SUPPORT,127140,NaN,NaN,NaN,NaN,NaN,NaN
392,tbd,Tuzinsky Consulting LLC,13741 Labelle St,NaN,Oak Park,MI,48237,Consulting,176325,NaN,NaN,NaN,NaN,NaN,NaN
393,tbd,MTAN Solutions LLC,966 Torke Terrace,NaN,Plymouth,MI,48170,Consulting,107250,NaN,NaN,NaN,NaN,NaN,NaN
394,tbd,Financial One Accounting Inc,44744 Helm St,NaN,Plymouth,MI,48170,Accounting Services,101076,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
pd.DataFrame(expense_data)

,EIN,ExpenseType,TotalAmt,ProgramServicesAmt,ManagementAndGeneralAmt,FundraisingAmt
0,tbd,Grants_DomesticGroup,0,0,NaN,NaN
1,tbd,Grants_DomesticIndividuals,0,0,NaN,NaN
2,tbd,Compensation_CurrentOfficersDirectors,10636915,6002286,4137346,497283
3,tbd,Fees_Lobbying,154123,0,154123,0
4,tbd,Fees_ProfessionalFundraising,300840,NaN,NaN,300840
...,...,...,...,...,...,...
2407,tbd,Payment_TravelForPublicOfficials,0,0,0,0
2408,tbd,Compensation_CurrentOfficersDirectors,74472,36268,38204,NaN
2409,tbd,Grants_DomesticGroup,2076596,2076596,NaN,NaN
2410,tbd,Grants_DomesticIndividuals,2900,2900,NaN,NaN


In [10]:
pd.DataFrame(balance_sheet_data)

,EIN,BalanceSheetSection,BalanceSheetSubsection,TotalAmt,ProgramServicesAmt,BOYAmt,EOYAmt
0,tbd,Assets,PledgesAndGrantsReceivable,0,0,NaN,NaN
1,tbd,Assets,TotalAssets,NaN,NaN,301872000,304229000
2,tbd,Liabilities,TotalLiabilities,NaN,NaN,88079000,77425000
3,tbd,NetAssetsOrFundBalances,NetAssets_Unrestricted,NaN,NaN,71082000,87606000
4,tbd,NetAssetsOrFundBalances,NetAssets_TemporarilyRestricted,NaN,NaN,90519000,85886000
...,...,...,...,...,...,...,...
4091,tbd,Liabilities,TotalLiabilities,NaN,NaN,0,0
4092,tbd,NetAssetsOrFundBalances,NetAssets_NoDonorRestrictions,NaN,NaN,210531,201416
4093,tbd,Assets,PledgesAndGrantsReceivable,82300,82300,NaN,NaN
4094,tbd,Assets,TotalAssets,NaN,NaN,80845,53390


Search files for instance that has target

In [ ]:
# NAMESPACE = '{http://www.irs.gov/efile}'
# RETURN_HEADER_PATH = f'./{NAMESPACE}ReturnHeader'
# PREPARER_FIRM_GROUP_PATH = f'./{NAMESPACE}PreparerFirmGrp'
# FILER_PATH = f'./{NAMESPACE}Filer'
# RETURN_DATA_PATH = f'./{NAMESPACE}ReturnData'
# IRS990_PATH = RETURN_DATA_PATH + f'/{NAMESPACE}IRS990'
# SCHEDULEA_PATH = f'./{NAMESPACE}IRS990ScheduleA'
# SCHEDULEC_PATH = f'./{NAMESPACE}IRS990ScheduleC'

# target = f'{NAMESPACE}OrganizationFollowsSFAS117Ind'

# for year_dir in xml_data_directory.iterdir():
#     for period_dir in year_dir.iterdir():
#         print(period_dir)
#         counter = 0
#         files_skipped = 0
#         for xml_file in period_dir.glob('*.xml'):
#             # Skip certain orgs and form types
#             root = ET.parse(xml_file).getroot()
#             org_type = get_org_type(root)
#             if org_type not in ['501c3', '501c4', '527']:
#                 files_skipped += 1
#                 continue

#             if '202011559349300711' in str(xml_file):
#                 continue

#             _990 = root.find(SCHEDULEC_PATH)
#             if _990 is not None:
#                 print(xml_file)
#                 raise TypeError
#             # target_text = _990.find(target).text
#             # if target_text not in ['false', '0', 'o', 'O']:
#             #     print(xml_file)
#             #     raise TypeError

xml\2020\2020_1A
xml\2020\2020_3A
xml\2024\2024_01A
xml\2024\2024_02A


KeyboardInterrupt: 

Search files for instance that does NOT have target

In [12]:
NAMESPACE = '{http://www.irs.gov/efile}'
RETURN_HEADER_PATH = f'./{NAMESPACE}ReturnHeader'
PREPARER_FIRM_GROUP_PATH = f'./{NAMESPACE}PreparerFirmGrp'
FILER_PATH = f'./{NAMESPACE}Filer'
RETURN_DATA_PATH = f'./{NAMESPACE}ReturnData'
IRS990_PATH = RETURN_DATA_PATH + f'/{NAMESPACE}IRS990'

target = f'{NAMESPACE}OrganizationFollowsSFAS117Ind'

for year_dir in xml_data_directory.iterdir():
    for period_dir in year_dir.iterdir():
        print(period_dir)
        counter = 0
        files_skipped = 0
        for xml_file in period_dir.glob('*.xml'):
            # Skip certain orgs and form types
            root = ET.parse(xml_file).getroot()
            org_type = get_org_type(root)
            if org_type not in ['501c3', '501c4', '527']:
                files_skipped += 1
                continue 
            
            if '202011559349301216' in str(xml_file):
                continue
            
            _990 = root.find(IRS990_PATH)
            try:
                _990.find(target).text
            except AttributeError:
                print(xml_file)
                raise AttributeError

xml\2020\2020_1A
xml\2020\2020_3A
xml\2020\2020_3A\202011559349300716_public.xml


AttributeError: 

In [13]:
data = []
counter = 0 
for year_dir in xml_data_directory.iterdir():
    for period_dir in year_dir.iterdir():
        print(period_dir)
        for xml_file in period_dir.glob('*.xml'):
            # Skip certain orgs and form types
            root = ET.parse(xml_file).getroot()
            org_type = get_org_type(root)
            if org_type not in ['501c3', '501c4', '527']:
                continue
            ReturnContent = {}
            # * Header
            Header = root.find(RETURN_HEADER_PATH)
            ReturnContent['EIN'] = Header.find(f'{NAMESPACE}Filer/{NAMESPACE}EIN').text
            ReturnContent['Name'] = Header.find(f'{NAMESPACE}Filer/{NAMESPACE}BusinessName/{NAMESPACE}BusinessNameLine1Txt').text
            ReturnContent['OrgType'] = org_type
            ReturnContent['file'] = str(xml_file)
            ReturnContent['TaxYr'] = Header.find(f'{NAMESPACE}TaxYr').text
            # * ReturnData
            ReturnData = root.find(RETURN_DATA_PATH)
            for section in ReturnData:
                ReturnContentKey = section.tag.replace(NAMESPACE, '')
                ReturnContent[ReturnContentKey] = 1
            data.append(ReturnContent)
            counter += 1
            if counter % 100 == 0:
                print(f'Parsed {counter} files.')

xml\2020\2020_1A
xml\2020\2020_3A
Parsed 100 files.
Parsed 200 files.
Parsed 300 files.
Parsed 400 files.
Parsed 500 files.
Parsed 600 files.
Parsed 700 files.
Parsed 800 files.
Parsed 900 files.
Parsed 1000 files.
Parsed 1100 files.
Parsed 1200 files.
Parsed 1300 files.
Parsed 1400 files.
Parsed 1500 files.
Parsed 1600 files.
Parsed 1700 files.
Parsed 1800 files.
Parsed 1900 files.
Parsed 2000 files.
Parsed 2100 files.
Parsed 2200 files.
Parsed 2300 files.
Parsed 2400 files.
Parsed 2500 files.
Parsed 2600 files.
Parsed 2700 files.
Parsed 2800 files.
Parsed 2900 files.
Parsed 3000 files.
Parsed 3100 files.
Parsed 3200 files.
Parsed 3300 files.
Parsed 3400 files.
Parsed 3500 files.
Parsed 3600 files.
Parsed 3700 files.
Parsed 3800 files.
Parsed 3900 files.
Parsed 4000 files.
Parsed 4100 files.
Parsed 4200 files.
Parsed 4300 files.
Parsed 4400 files.
Parsed 4500 files.
Parsed 4600 files.
Parsed 4700 files.
Parsed 4800 files.
Parsed 4900 files.
Parsed 5000 files.
Parsed 5100 files.
Parsed

In [ ]:
filing_content_matrix = pd.DataFrame(data)
filing_content_matrix.loc[:, 'file'] = filing_content_matrix.loc[:, 'file'].apply(lambda x: '\\'.join(x.split('\\')[2:]))
filing_content_matrix[filing_content_matrix.isna()] = 0
cols_to_convert = [col for col in filing_content_matrix.columns if col not in ['EIN', 'Name', 'OrgType', 'file']]
for col in cols_to_convert:
    filing_content_matrix[col] = filing_content_matrix[col].astype(int)
filing_content_matrix.to_csv('filing_content_matrix.csv')

In [25]:
filing_content_matrix

,EIN,Name,OrgType,file,TaxYr,IRS990,IRS990ScheduleA,IRS990ScheduleC,IRS990ScheduleD,IRS990ScheduleG,...,IRS990ScheduleB,IRS990ScheduleI,IRS990ScheduleF,AffiliatedGroupAttachment,IRS990ScheduleE,ReasonableCauseExplanation,IRS990ScheduleN,AveragingAttachment,AffiliatedGroupSchedule,AffiliateListing
0,742044647,NATIONAL JEWISH HEALTH,501c3,xml\2020\2020_3A\202011559349300711_public.xml,2018,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,581416325,HAWTREE VOLUNTEER FIRE DEPARTMENT INC,501c3,xml\2020\2020_3A\202011559349300716_public.xml,2019,1,1,0,1,0,...,1,0,0,0,0,0,0,0,0,0
2,223139262,RURAL DEVELOPMENT INC,501c3,xml\2020\2020_3A\202011559349300726_public.xml,2018,1,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,363244264,WILL COUNTY GOVERNMENTAL LEAGUE,501c4,xml\2020\2020_3A\202011559349300731_public.xml,2018,1,0,1,0,1,...,1,0,0,0,0,0,0,0,0,0
4,043316917,I'M STILL HERE FOUNDATION INC,501c3,xml\2020\2020_3A\202011559349300736_public.xml,2019,1,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413994,431556293,CIEFA MINISTRY INTERNATIONAL,501c3,xml\2026\2026_4A\202641139349301959_public.xml,2025,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
413995,260637571,ARMS OF CARE INTERNATIONAL INC,501c3,xml\2026\2026_4A\202641139349302004_public.xml,2025,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
413996,382878506,Christ Centered Homes Inc,501c3,xml\2026\2026_4A\202641139349302009_public.xml,2024,1,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
413997,330416470,HABITAT FOR HUMANITY OF GREATER,501c3,xml\2026\2026_4A\202641139349302054_public.xml,2024,1,1,0,1,1,...,1,0,0,0,0,0,0,0,0,0
